# Native crypto L3 capture and replay

**Activity: market-data fetching and offline inspection.** This notebook is the
bounded MRs 22–27 learning checkpoint. It calls installed package behavior and
records real Bitfinex R0 input. Coinbase uses authenticated Exchange `level3`
when the local Exchange key is available; otherwise the exact access blocker is
shown and independent synthetic lifecycle checks still run.

We will connect, persist, decode, inspect gaps, and replay saved bytes. The
result is visible resting orders within source coverage. It does not establish
queue priority, hidden liquidity or executable fills. Live recording creates new
history; the Binance historical workflow remains in `research-foundations.ipynb`.
See [the collector guide](../docs/data/l3-collectors.md) for protocol references,
limits, credentials, CLI commands and failure handling.

## 1. Installed environment and explicit source selection

Run all cells with a fresh kernel through `examples/run_notebook.py`. No cell
changes `sys.path`, installs dependencies, prints credentials or starts a service.
Package source and lock identities distinguish this execution from prior runs.

In [1]:
import hashlib
import json
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

import alpha_research
import pandas as pd

from alpha_research.data.live.config import CaptureConfig, CaptureRequest
from alpha_research.data.live.journal import CaptureJournal, CaptureRecord
from alpha_research.data.live.replay import CaptureReplay
from alpha_research.data.live.transport import VenueTransport
from alpha_research.data.venues.coinbase.auth import CoinbaseAccessError, CoinbaseCredentials
from alpha_research.data.venues.coinbase.book import CoinbaseBook
from alpha_research.pipelines.live import InstalledIdentity, collect
from alpha_research.serialization import DomainCodec

root = Path.cwd()
assert (root / "configs/live").is_dir(), "Run from the repository root"
codec = DomainCodec()
implementation = InstalledIdentity.sha256()
bitfinex = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
coinbase = CaptureConfig.read(root / "configs/live/coinbase-btc-usd.json")
pd.DataFrame([{"python": sys.version.split()[0], "package": alpha_research.__file__,
               "implementation_sha256": implementation,
               "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest()}])

,python,package,implementation_sha256,lock_sha256
0,3.14.7,/Users/lxg/Documents/GitHub/Alpha Research/alp...,b57d3a13b900aef2d1a55a837140a9d4c140e5126ba3d3...,e6cd3a169ff4b6850bcb5b09e38c44e1179b0dc1fb7c6a...


## 2. Native instruments and connection requirements

`probe` checks current public listing availability before collection. These are
native BTC/USD spot listings with separate canonical identities. Their public
availability does not prove that this account can subscribe to Exchange Level 3.
Bitfinex is public; Coinbase requires Exchange key/secret/passphrase. CDP JSON is
ignored. Configured limits below bound storage, message receipt and state.

In [2]:
discovery = []
for config in (bitfinex, coinbase):
    payload = VenueTransport(config).discover()
    discovery.append({"venue": config.venue, "native_symbol": config.symbol,
                      "instrument_id": str(config.instrument_id), "channel": config.channel,
                      "duration_s": config.duration_seconds, "message_limit": config.max_messages,
                      "journal_bytes": config.max_bytes, "depth": config.depth,
                      "discovery_sha256": hashlib.sha256(payload).hexdigest()})
pd.DataFrame(discovery)

,venue,native_symbol,instrument_id,channel,duration_s,message_limit,journal_bytes,depth,discovery_sha256
0,bitfinex,tBTCUSD,c87d983f-e7ec-4e64-8d54-9f2be67a64b8,R0,20,10000,32000000,25,ad5c2c16d99b75060bdfcb43aa70faa233931623d8b953...
1,coinbase,BTC-USD,218142de-d549-43bb-85ba-a174fd4f798f,level3,20,10000,32000000,0,d2bfd4363396ca895bed3a49a2e3f265c7c043efbb50b0...


## 3. Real bounded recording

The receiver timestamps each complete frame and sends it to a bounded queue.
The writer fsyncs each journal line before decoding it. Acknowledgements,
heartbeats, schema frames and epoch boundaries are retained alongside market
frames. A UUID directory prevents same-day overwrite. Gap/retry reasons remain
visible even when a later epoch recovers. Each call ends after its configured
limits; nothing runs unattended after this cell.

In [3]:
bitfinex_result = await collect(bitfinex, root / "data/bitfinex",
                                implementation_identity=implementation)
assert bitfinex_result["replay_parity"]
assert bitfinex_result["synchronized"] and not bitfinex_result["gaps"], bitfinex_result
bitfinex_result

{'venue': 'bitfinex',
 'symbol': 'tBTCUSD',
 'instrument_id': 'c87d983f-e7ec-4e64-8d54-9f2be67a64b8',
 'channel': 'R0',
 'started_ns': 1789516373341207000,
 'finished_ns': 1789516393532686000,
 'messages': 363,
 'updates': 358,
 'snapshots': 1,
 'heartbeats': 1,
 'orders': 50,
 'depth': 25,
 'max_orders': 100000,
 'gaps': [],
 'synchronized': True,
 'ended': True,
 'source_sequence': 360,
 'last_source_time': None,
 'book_identity': '009da6d245eb322aa6e848d6aaed70afdce43b36ec1516fb473ad0f422fef76c',
 'trace_identity': '9de4b387a87aac30db11ed363ea23ea109ab01a80d0c433e7628c1141432562b',
 'epochs': 1,
 'request_identity': '3a911dcc5b7b63b80b783d9800993fb69e2553aa7dd1b632a45f9ee65fcf83b1',
 'saved_input_identity': 'a57a31aef5ca719a3cbaebf951b4ece0644b1f8ae067b3120456430bd6d57041',
 'saved_bytes': 50714,
 'capture_directory': '/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/bitfinex/3e061476-ccca-41a1-b09f-8b3010a75013',
 'stop_reason': 'duration_limit',
 'replay_

## 4. Coinbase access boundary

Fill `secrets/coinbase.secrets` locally using the three placeholders in the guide.
The loader rejects empty or malformed fields before opening a connection.
If eligible credentials are available this cell runs the same bounded collector;
otherwise it reports access-blocked status and continues to independent checks.
Credential values and signed outbound subscriptions are never displayed.

In [4]:
coinbase_result = None
try:
    exchange_credentials = CoinbaseCredentials.read(root / "secrets/coinbase.secrets")
except CoinbaseAccessError as error:
    coinbase_access = {"venue": "coinbase", "status": "access_blocked", "reason": str(error)}
else:
    coinbase_result = await collect(coinbase, root / "data/coinbase",
                                   implementation_identity=implementation,
                                   credentials=exchange_credentials)
    del exchange_credentials
    coinbase_access = {"venue": "coinbase", "status": "captured",
                       "synchronized": coinbase_result["synchronized"],
                       "gaps": coinbase_result["gaps"]}
coinbase_access

{'venue': 'coinbase',
 'status': 'access_blocked',
 'reason': 'Fill COINBASE_API_KEY, COINBASE_API_SECRET (base64), and COINBASE_API_PASSPHRASE in the Exchange secrets file'}

## 5. Meaningful interval and persisted-input identity

Receipt nanoseconds are UTC application observations, preserved without sorting.
Bitfinex supplies no source timestamp under the selected flags, so it remains
unknown. Depth 25 describes the requested raw-book window, not all venue orders.
Counts and saved identities below are derived from this run, not prefilled values.

In [5]:
results = [bitfinex_result] + ([] if coinbase_result is None else [coinbase_result])
rows = []
for result in results:
    row = {name: result[name] for name in ("venue", "symbol", "messages", "snapshots", "updates",
                                          "orders", "depth", "gaps", "saved_input_identity")}
    for name in ("started_ns", "finished_ns"):
        row[name.replace("_ns", "_utc")] = (
            datetime(1970, 1, 1, tzinfo=UTC) + timedelta(microseconds=result[name] // 1000)
        ).isoformat()
    rows.append(row)
pd.DataFrame(rows)

,venue,symbol,messages,snapshots,updates,orders,depth,gaps,saved_input_identity,started_utc,finished_utc
0,bitfinex,tBTCUSD,363,1,358,50,25,[],a57a31aef5ca719a3cbaebf951b4ece0644b1f8ae067b3...,2026-09-15T23:52:53.341207+00:00,2026-09-15T23:53:13.532686+00:00


## 6. Inspect persistence and exact decoding

The manifest uses the existing raw-store SHA-256 identity. Reads verify all
published bytes before decoding. We display the receipt envelope and numeric
**types**, keeping bulk raw feed data under the ignored venue directory. Source
order IDs and decimal tokens are preserved in saved bytes. The replay consumer
uses the same venue decoder as collection.

In [6]:
capture_root = Path(bitfinex_result["capture_directory"])
request = codec.decode_json((capture_root / "request.json").read_bytes(), CaptureRequest)
journal = CaptureJournal(capture_root, request)
first_frame = next(record for record in journal.records() if record.kind == "frame")
offline = CaptureReplay.read(capture_root)
example_order = next(iter(offline.book.orders.values()))
pd.DataFrame([{"epoch": first_frame.epoch, "receipt_ns": first_frame.received_ns,
               "raw_frame_bytes": len(first_frame.raw),
               "raw_frame_sha256": hashlib.sha256(first_frame.raw).hexdigest(),
               "order_id_type": type(example_order.order_id).__name__,
               "price_type": type(example_order.price).__name__,
               "size_type": type(example_order.size).__name__,
               "source_time": offline.book.source_time}])

,epoch,receipt_ns,raw_frame_bytes,raw_frame_sha256,order_id_type,price_type,size_type,source_time
0,0,1789516373933462000,102,9fdfc92db0d3000b155fe6cf285871919d033c8eadd071...,str,Decimal,Decimal,None


## 7. Offline replay and a synthetic gap exercise

No network is needed to reopen `capture_root`. The rolling trace compares state
at every source-message boundary, including controls and receipt clocks. The
final state and trace must equal collection. A checksum identity alone does not
make missing input retrievable: keep the local directory together with its
manifest and request file.

**Exercise:** append a synthetic disconnect to the observed prefix. Does a later
heartbeat alone make the book usable? The answer is no: the affected epoch stays
invalid until a fresh subscription and snapshot. This exercise does not alter
the real saved input.

In [7]:
assert offline.summary()["trace_identity"] == bitfinex_result["trace_identity"]
assert offline.summary()["book_identity"] == bitfinex_result["book_identity"]
assert offline.messages == bitfinex_result["messages"]
exercise = CaptureReplay(bitfinex)
for record in journal.records():
    if record.kind == "end":
        break
    exercise.apply(record)
exercise.apply(CaptureRecord(kind="gap", epoch=exercise.epoch,
                             received_ns=exercise.finished_ns, reason="synthetic_disconnect"))
assert not exercise.book.ready
{"offline_parity": True, "verified_messages": offline.messages,
 "trace_sha256": offline.summary()["trace_identity"],
 "synthetic_gap": exercise.gaps[-1], "usable_after_gap": exercise.book.ready}

{'offline_parity': True,
 'verified_messages': 363,
 'trace_sha256': '9de4b387a87aac30db11ed363ea23ea109ab01a80d0c433e7628c1141432562b',
 'synthetic_gap': {'epoch': 0,
  'received_ns': 1789516393529767000,
  'reason': 'synthetic_disconnect'},
 'usable_after_gap': False}

## 8. Coinbase snapshot bridge and lifecycle, independent of account access

This final case is explicitly synthetic. It uses the official compact schema
shape and a hand-authored full L3 snapshot at sequence 10. A queued `noop` at 11
bridges the snapshot; then a maker match at 12 reduces exact resting size.
Received/non-resting events must not be treated as visible opens. Synthetic
checks establish software behavior; they do not establish authenticated source
access, queue priority or hypothetical execution quality.

In [8]:
coinbase_book = CoinbaseBook(coinbase)
coinbase_book.apply((root / "tests/fixtures/l3/coinbase-schema.json").read_bytes())
coinbase_book.apply(b'{"type":"subscriptions","channels":[{"name":"level3","product_ids":["BTC-USD"]}]}')
coinbase_book.apply(b'["noop","BTC-USD","11",null]')
coinbase_book.apply((root / "tests/fixtures/l3/coinbase-snapshot.json").read_bytes(), snapshot=True)
coinbase_book.apply(b'["match","BTC-USD","12","maker","taker","100","0.1",null]')
assert str(coinbase_book.orders["maker"].size) == "0.200000000000000001"
assert coinbase_book.ready
{"evidence": "synthetic", "sequence": coinbase_book.sequence,
 "remaining_size": str(coinbase_book.orders["maker"].size),
 "orders": len(coinbase_book.orders), "source_time": coinbase_book.source_time,
 "authenticated_live_evidence": coinbase_access["status"]}

{'evidence': 'synthetic',
 'sequence': 12,
 'remaining_size': '0.200000000000000001',
 'orders': 1,
 'source_time': None,
 'authenticated_live_evidence': 'access_blocked'}

## Interpretation and stopping point

The Bitfinex success path above is a real bounded capture with verified offline
parity. Coinbase live status is stated separately. Malformed schemas, numeric
errors, missing auth, bridge/sequence gaps, disconnect/recovery, queue overflow,
byte limits and interrupted persistence also have maintained deterministic tests.
The current reconstruction supports visible order observations only. Canonical
dataset publication, generalized replay scheduling, external supervision and all
execution modeling remain with their roadmap owners. Close the kernel when done.